# 第 8 周练习：代码生成的双代理工作流

## 练习目标

本笔记本演示 **两个协作代理**（Senior Architect + Senior Developer）如何把自然语言需求变成可下载的代码包：

1. **架构师代理** — 用 JSON 模式规划需要哪些文件及各自职责  
2. **高级开发代理** — 按计划逐文件生成 Ruby on Rails 代码  
3. **打包** — 把生成结果打成 ZIP，经 Gradio UI 下载  

## 和本课 Week 8 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 多代理编排 | 架构师 → 开发者 流水线 |
| 结构化输出 | `response_format` JSON 模式 |
| Gradio 部署界面 | 查询框 + 日志 + 文件下载 |

## 怎么跑

1. 在 `.env` 中设置 `OPENAI_API_KEY`  
2. 按顺序运行全部单元格  
3. 在 Gradio 里输入功能描述，等待 ZIP 生成  


In [ ]:
# ========== 导入、环境变量与系统提示词（Prompt 原文勿改）==========

# os：读 API Key；路径相关操作在后续单元格也会用到
import os
# json：解析架构师返回的文件计划
import json
# zipfile：把生成目录打成 ZIP 供下载
import zipfile
# shutil：清理上次生成的工作区目录
import shutil
# Gradio：浏览器端交互 UI
import gradio as gr
# OpenAI 官方客户端（Chat Completions）
from openai import OpenAI
# 从 .env 加载密钥，避免把密钥写进代码
from dotenv import load_dotenv

# override=True：用 .env 覆盖已有同名环境变量
load_dotenv(override=True)
# 从环境变量读取 OpenAI API Key
api_key = os.getenv('OPENAI_API_KEY')

# 本练习使用的模型 id（保持原样，勿改）
MODEL = "gpt-5.4-mini"

# 架构师系统提示：要求输出含 files 数组的 JSON（prompt 英文保留）
SENIOR_ARCHITECT_AGENT_SYSTEM_PROMPT = """
You are an expert Ruby on Rails software architect.
Given a user's feature request, determine the necessary files to build it.
Output a JSON object containing a 'files' array. Each item in the array must have a 'filename' (string) and a 'description' (string detailing the specific code logic needed)."""

# 开发者系统提示：只要原始代码，不要 markdown 围栏（prompt 英文保留）
SENIOR_DEVELOPER_AGENT_SYSTEM_PROMPT = """
You are Ruby on Rails senior developer.
Write the code for the requested file based on the description.
Return ONLY the raw code. Do not wrap the code in markdown blocks (e.g., ```python) and do not include any conversational text."""


In [ ]:
# ========== 架构师代理：用 JSON 模式产出文件计划 ==========

def architect_agent(query, logs):
    # 每次调用新建客户端（使用全局 api_key）
    client = OpenAI(api_key=api_key)
    # Chat Completions：强制 json_object，便于后续 json.loads
    plan_response = client.chat.completions.create(
        model=MODEL,
        response_format={ "type": "json_object" },
        messages=[
            {
                "role": "system",
                "content": SENIOR_ARCHITECT_AGENT_SYSTEM_PROMPT
            },
            # 用户的功能需求原文作为 user 消息
            {"role": "user", "content": query}
        ]
    )

    # 取出模型返回的 JSON 字符串
    plan_content = plan_response.choices[0].message.content
    # 解析为 Python dict
    plan_json = json.loads(plan_content)
    # 取出 files 列表；缺失时用空列表兜底
    files_to_create = plan_json.get("files", [])

    # 把计划摘要追加到日志（给 Gradio 展示）
    logs += f"Plan created! {len(files_to_create)} files to generate.\n\n"
    for f in files_to_create:
        # 每条只展示描述前 50 字符，避免日志过长
        logs += f" - {f['filename']}: {f['description'][:50]}...\n"

    # 返回：待生成文件列表 + 更新后的日志
    return files_to_create, logs


In [ ]:
# ========== 高级开发代理：按文件名+描述写出代码并落盘 ==========

def seniordev_agent(filename, description, output_dir):
  # 为本次代码生成创建 OpenAI 客户端
  client = OpenAI(api_key=api_key)

  # 调用模型：系统提示约束「只返回原始代码」
  code_response = client.chat.completions.create(
      model=MODEL,
      messages=[
          {
              "role": "system",
              "content": SENIOR_DEVELOPER_AGENT_SYSTEM_PROMPT
          },
          {
              "role": "user",
              # 把文件名与职责描述一并交给模型
              "content": f"Filename: {filename}\nDescription: {description}"
          }
      ]
  )

  # 取出生成正文
  raw_code = code_response.choices[0].message.content

  # 故障安全：若仍包了 ``` 围栏，剥掉首尾行
  if raw_code.startswith("```"):
      lines = raw_code.split("\n")
      raw_code = "\n".join(lines[1:-1])

  # 拼出目标路径：output_dir / filename
  file_path = os.path.join(output_dir, filename)

  # 若路径含子目录，先确保父目录存在（dirname 为空时用 "."）
  os.makedirs(os.path.dirname(file_path) or ".", exist_ok=True)

  # 以 UTF-8 写入生成代码
  with open(file_path, "w", encoding="utf-8") as f:
      f.write(raw_code)


In [ ]:
# ========== Gradio 回调：计划 → 写代码 → 打 ZIP ==========

def generate_and_zip(query):
    # 本次生成的工作目录名
    output_dir = "generated_workspace"
    # 最终提供下载的 ZIP 文件名
    zip_filename = "generated_feature.zip"
    # 初始日志（英文文案保留，供 UI 显示）
    logs = "Starting generation process...\n\n"

    # 清理上次残留的工作区，避免混入旧文件
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    # 新建空工作区
    os.makedirs(output_dir)

    # 若旧 ZIP 仍在，先删除
    if os.path.exists(zip_filename):
        os.remove(zip_filename)

    # —— 步骤 1：架构师制定文件计划 ——
    try:
        logs += "1. Generating project plan...\n"
        files_to_create, logs = architect_agent(query, logs)
    except Exception as e:
        # 规划失败：返回错误字符串，无文件
        return f"Error during planning phase: {str(e)}", None

    # —— 步骤 2：开发代理逐文件写代码 ——
    logs += "\n2. Writing code for files...\n"

    for file_info in files_to_create:
        # 从计划项取出文件名与描述
        filename = file_info['filename']
        description = file_info['description']
        logs += f"   -> Generating {filename}...\n"

        try:
            # 调用开发代理落盘
            seniordev_agent(filename, description, output_dir)
        except Exception as e:
            # 单文件失败不中断整条流水线，只记日志
            logs += f"   -> Error generating {filename}: {str(e)}\n"

    # —— 步骤 3：遍历工作区，打成 ZIP ——
    logs += "\n3. Zipping files...\n"
    try:
        with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for root, dirs, files in os.walk(output_dir):
                for file in files:
                    file_path = os.path.join(root, file)
                    # ZIP 内使用相对路径，避免绝对路径泄漏
                    arcname = os.path.relpath(file_path, output_dir)
                    zipf.write(file_path, arcname)
        logs += "\nProcess Complete! You can download your ZIP file below."
    except Exception as e:
        return f"Error zipping files: {str(e)}", None

    # 成功：返回日志文本 + ZIP 路径（Gradio File 组件可下载）
    return logs, zip_filename


In [ ]:
# ========== 构建 Gradio Blocks UI 并启动 ==========

# Soft 主题的 Blocks 应用
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    # 顶部说明（UI 字符串保持原样，避免改动展示逻辑）
    gr.Markdown(
        """
        # 🚄 Ruby on Rails 专家代理
        Enter what you want to build. The AI Agents will formulate a plan, write the files, and give you a ZIP containing the final codebase.
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            # 左侧：功能需求输入框
            query_input = gr.Textbox(
                label="Feature Query",
                lines=5,
                placeholder="E.g. Create an online Shop with Products, Categories, Orders and Customers including a REST API."
            )
            # 主按钮：触发 generate_and_zip
            generate_btn = gr.Button("Generate & Download Code", variant="primary")

        with gr.Column(scale=1):
            # 右侧：执行日志与计划
            log_output = gr.Textbox(label="Execution Logs & Plan", lines=12, interactive=False)
            # 下载生成的 ZIP
            file_output = gr.File(label="Download Generated Source Code")

    # 把按钮点击接到回调：输入 query → 输出日志 + 文件
    generate_btn.click(
        fn=generate_and_zip,
        inputs=[query_input],
        outputs=[log_output, file_output]
    )

# 在 Jupyter 中启动；debug=True 便于看报错；inbrowser=True 自动开浏览器
demo.launch(debug=True, inbrowser=True)
